# 02 - Regex features from complaint narratives

This notebook cleans narrative text and extracts simple, explainable indicators. The five example narratives below are **synthetic** and contain no real consumer information. The first amount found is saved in `amount_mentioned`; all found amount/date strings are also kept for review. Regex matches are clues for analysis, not proof of loss or fraud.

The reusable implementation lives in `src/regex_features.py`. The notebook demonstrates the functions on five examples, explains each pattern family, and includes an optional chunked run for the CFPB CSV.

In [ ]:
# Import the standard libraries and pandas used to display the example feature table.
from pathlib import Path
import sys

import pandas as pd

# Make the project source folder importable when Jupyter starts in the project root.
project_root = Path.cwd()
if not (project_root / "src").is_dir():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

# Load the same reusable feature functions used by the command-line script.
from src.regex_features import add_regex_features, process_csv

## Five before/after examples

The **before** column is the original synthetic narrative. The **after** column is whitespace-normalized plain text; the remaining columns show what the regex extractor found. A blank amount means no explicitly dollar-marked amount was detected.

| # | Before (synthetic) | After cleanup |
|---|---|---|
| 1 | `On 03/14/2024 I was charged $1,250.00 for an account fee. My card ending XXXX-1234 was used.` | `On 03/14/2024 I was charged $1,250.00 for an account fee. My card ending XXXX-1234 was used.` |
| 2 | `<p>My debit card showed a $45.99 charge on January 5, 2025.</p>   I did not authorize it.` | `My debit card showed a $45.99 charge on January 5, 2025. I did not authorize it.` |
| 3 | `The account number ending in **** 6789 was billed USD 2,000 and again $35.` | `The account number ending in **** 6789 was billed USD 2,000 and again $35.` |
| 4 | `I called about my credit card on 2024-11-02; no dollar value is listed.` | `I called about my credit card on 2024-11-02; no dollar value is listed.` |
| 5 | `No masked number was provided; bank account still locked.` | `No masked number was provided; bank account still locked.` |

In [ ]:
# Use fabricated examples so the notebook always demonstrates the patterns safely.
examples = pd.DataFrame(
    {
        "Consumer complaint narrative": [
            "On 03/14/2024 I was charged $1,250.00 for an account fee. My card ending XXXX-1234 was used.",
            "<p>My debit card showed a $45.99 charge on January 5, 2025.</p>   I did not authorize it.",
            "The account number ending in **** 6789 was billed USD 2,000 and again $35.",
            "I called about my credit card on 2024-11-02; no dollar value is listed.",
            "No masked number was provided; bank account still locked.",
        ]
    }
)

# Add the features, then select a compact set of columns for the before/after view.
example_features = add_regex_features(examples)
example_features[
    [
        "Consumer complaint narrative",
        "narrative_clean",
        "amount_mentioned",
        "amount_mentions",
        "date_mentions",
        "has_account_card_mention",
        "n_masked_tokens",
        "narrative_length",
    ]
]

## Pattern explanations

- **Narrative cleanup:** HTML-like tags are replaced with spaces, HTML entities are decoded, control characters are removed, and repeated whitespace is collapsed. This retains the narrative wording; it is not a de-identification step.
- **Dollar amounts:** `(?<![\\w.])(?:USD\\s*|\\$\\s*)(\\d{1,3}(?:,\\d{3})+|\\d+)(?:\\.(\\d{1,2}))?(?![\\w,]|\\.\\d)` requires an explicit `$` or `USD`, supports comma-separated thousands and up to two decimal digits, and avoids partial matches within words, malformed comma groups, and longer decimal values. It allows a sentence-ending period after an amount. `amount_mentioned` stores the first match as a number; `amount_mentions` and `n_amount_mentions` retain the matched values and count. Amounts without a currency marker are intentionally not guessed.
- **Dates:** `\\d{4}-\\d{1,2}-\\d{1,2}`, `\\d{1,2}[/-]\\d{1,2}[/-]\\d{2,4}`, and month-name alternatives find common ISO, slash/dash numeric, and English month-name formats. `date_mentions`, `n_date_mentions`, and `has_date` expose the matches. The patterns detect date-like strings but do not validate calendar correctness or interpret ambiguous day/month order.
- **Account/card mentions:** case-insensitive word-boundary patterns look for `account`/`acct` and `card`/`credit card`/`debit card`; they set individual flags and the combined `has_account_card_mention` flag. A mention does not mean a number was exposed.
- **Masked PII tokens:** `(?<![A-Za-z0-9])(?:X{2,}|[*#•]{2,})(?![A-Za-z0-9])` counts visible runs such as `XXXX`, `****`, or `####`, without trying to reconstruct the hidden digits. Each run is one token.
- **Narrative length:** `narrative_length` is the number of characters after cleanup, not a word or token count.

## Optional: process the CFPB CSV

Place the downloaded source at `data/raw/complaints.csv`, then run the next cell to process it in chunks and write `data/processed/complaints_regex_features.csv`. Chunking limits memory use. This reads and writes the original columns as well as the added features. Keep raw consumer data protected and do not commit the source CSV.

In [ ]:
# Only run the full-dataset step when the source CSV is available locally.
input_csv = project_root / "data" / "raw" / "complaints.csv"
output_csv = project_root / "data" / "processed" / "complaints_regex_features.csv"

if input_csv.is_file():
    processed_rows = process_csv(input_csv, output_csv, chunksize=50_000)
    print(f"Wrote {processed_rows:,} rows to {output_csv}")
else:
    print(f"CSV not found yet: {input_csv}. Run the synthetic examples above first.")